In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from spectra.utilities import read_OE65PRO_spectrum_data_file, find_index_of_nearest

In [2]:
lightsource_spectrometer_file = Path('raw_data') / 'QEPB00791_17-07-21-669.txt'

lightsource_spectrometer_header, lightsource_spectrometer_data = read_OE65PRO_spectrum_data_file(lightsource_spectrometer_file)
lightsource_spectrometer_data.shape

raw_data/QEPB00791_17-07-21-669.txt


(1044, 2)

In [3]:
PEGDA_spectrometer_file = Path('raw_data') / 'QEPB00791_17-10-07-458.txt'

PEGDA_spectrometer_header, PEGDA_spectrometer_data = read_OE65PRO_spectrum_data_file(PEGDA_spectrometer_file)
PEGDA_spectrometer_data.shape

raw_data/QEPB00791_17-10-07-458.txt


(1044, 2)

In [4]:
avobenzone_spectrometer_file = Path('raw_data') / 'QEPB00791_17-18-57-140.txt'

avobenzone_spectrometer_header, avobenzone_spectrometer_data = read_OE65PRO_spectrum_data_file(avobenzone_spectrometer_file)
avobenzone_spectrometer_data.shape

raw_data/QEPB00791_17-18-57-140.txt


(1044, 2)

In [5]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(PEGDA_spectrometer_data[:,0], PEGDA_spectrometer_data[:,1], label='PEGDA')
ax.plot(avobenzone_spectrometer_data[:,0], avobenzone_spectrometer_data[:,1], label='avobenzone')
# ax.plot(lightsource_spectrometer_data[:,0], lightsource_spectrometer_data[:,1], label='Light Source')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Signal')
ax.set_title("Raw Spectrum")
# ax.set_xlim([320, 460])
# ax.set_ylim([0, 60000])
ax.grid()
ax.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Restrict data sets to 300-540 nm

Reason:
- Low end - arbitrary
- High end - just between 532 nm laser wavelength and saturated spectrum values.

In [6]:
# Make sure both data sets are for the same wavelengths
assert np.allclose(PEGDA_spectrometer_data[:,0], avobenzone_spectrometer_data[:,0])

In [59]:
def find_indices_of_range(arr, low, high):
    assert len(arr.shape) == 1
    
    index_low = find_index_of_nearest(arr, low)
    index_high = find_index_of_nearest(arr, high)
    
    return index_low, index_high


def select_wavelength_range(data, wavelength_low, wavelength_high):
    assert len(data.shape) == 2
    assert data.shape[1] == 2
    
    index_low, index_high = find_indices_of_range(data[:,0], wavelength_low, wavelength_high)
    
    return data[index_low:index_high+1]


def calc_absorbance(source_spectrum, absorber_spectrum, wavelength_low, wavelength_high, force_positive=True, num_samples_for_average=10):
    
    temp_source = select_wavelength_range(source_spectrum, wavelength_low, wavelength_high)
    temp_absorber = select_wavelength_range(absorber_spectrum, wavelength_low, wavelength_high)
    
    absorbance = np.column_stack((temp_source[:,0], -np.log10(temp_absorber[:,1] / temp_source[:,1])))
    
    if force_positive:
        avg = np.mean(absorbance[-num_samples_for_average:-1, 1])
        if avg < 0.0:
            absorbance[:, 1] += -avg
        
    return absorbance

In [60]:
# Test select_wavelength_range()

temp = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
    [4, 40],
    [5, 50],
    [6, 60],
])

temp2 = select_wavelength_range(temp, 2.1, 4.9)
temp2

array([[ 2, 20],
       [ 3, 30],
       [ 4, 40],
       [ 5, 50]])

In [61]:
# Test absorbance

temp1 = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
    [4, 40],
    [5, 50],
    [6, 60],
])

temp2 = np.array([
    [1, 1],
    [2, 19],
    [3, 3],
    [4, 38],
    [5, 5],
    [6, 57],
])

calc_absorbance(temp1, temp2, 2.1, 4.9, False)

array([[2.        , 0.02227639],
       [3.        , 1.        ],
       [4.        , 0.02227639],
       [5.        , 1.        ]])

In [62]:
# Test absorbance

temp1 = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
    [4, 40],
    [5, 50],
    [6, 60],
])

temp2 = np.array([
    [1, 1],
    [2, 22],
    [3, 3],
    [4, 44],
    [5, 55],
    [6, 66],
])

print(calc_absorbance(temp1, temp2, 2.1, 4.9, False))
print()
print(calc_absorbance(temp1, temp2, 2.1, 4.9, True, num_samples_for_average=2))

[[ 2.         -0.04139269]
 [ 3.          1.        ]
 [ 4.         -0.04139269]
 [ 5.         -0.04139269]]

-0.04139268515822508
[[2.         0.        ]
 [3.         1.04139269]
 [4.         0.        ]
 [5.         0.        ]]


# Calculate absorbance

In [63]:
wavelength_range = [300, 540]

absorbance = calc_absorbance(PEGDA_spectrometer_data, avobenzone_spectrometer_data, *wavelength_range)

-0.02865188383632147


In [64]:
fig, ax = plt.subplots()
ax.plot(absorbance[:,0], absorbance[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Signal')
ax.set_title("Avobenzone Absorbance")
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
wavelength_range = [300, 540]

index_low = find_index_of_nearest(PEGDA_spectrometer_data[:,0], wavelength_range[0])
index_high = find_index_of_nearest(PEGDA_spectrometer_data[:,0], wavelength_range[1])

print(index_low, PEGDA_spectrometer_data[index_low, :])
print(index_high, PEGDA_spectrometer_data[index_high, :])